# 10 · Transformer Mimarisi: Decoder Bloğu

**Belgedeki bölüm:** Transformer Mimarisi (Encoder - Decoder) › Decoder Katmanı, GPT Gibi Modeller

- Belgedeki şema encoder–decoder; GPT gibi modeller (ve bu kurstaki UstaModel) **sadece decoder** kullanır, cross-attention yoktur.
- Bir decoder katmanı: Causal Multi-Head Attention → Add & Norm → Feed-Forward (MLP) → Add & Norm.
- Feed-Forward: her token kendi içinde önce 4 kat genişler (768 → 3072), sonra eski boyutuna döner; araya GELU aktivasyonu girer.

In [ ]:
import torch
import torch.nn as nn

![transformer-architecture](https://deeprevision.github.io/posts/001-transformer/transformer.png)

![gelu](https://miro.medium.com/v2/resize:fit:4800/format:webp/1*O5E-huBuY1UTHMmM--rhLQ.png)

In [ ]:
import torch.nn as nn

class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (
      1 + torch.tanh(
          torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        )
    )


gelu = GELU()

example_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
gelu(example_tensor)

In [ ]:
# import torch funtions 

import torch.nn.functional as F

F.gelu(example_tensor, approximate="tanh")

FFN de matrislerde daha iyi işlem yapabilmek için matris genişletme işlemleri

In [ ]:
class UstaMLP(nn.Module):
  def __init__(self, embedding_dim, hidden_dim):
    super().__init__()

    self.gate_proj = nn.Linear(embedding_dim, hidden_dim)
    self.up_proj = nn.Linear(embedding_dim, hidden_dim)
    self.down_proj = nn.Linear(hidden_dim, embedding_dim)
    self.gelu = GELU()

  def forward(self, x):
    """ gate = self.gate_proj(x)
        gate = F.gelu(gate, approximate="tanh")
        up = self.up_proj(x)
        fuse = gate * up
        outputs = self.down_proj(fuse) """
    gate = self.gate_proj(x)
    gate = self.gelu(gate)
    up = self.up_proj(x)
    fuse = gate * up
    outputs = self.down_proj(fuse)
    return outputs

In [ ]:
mlp = UstaMLP(embedding_dim=4, hidden_dim=16)  # 4 -> 16 -> 4 (4 kat genişleme)
mlp(torch.randn(3, 4)).shape

### Decoder bloğu = MHA → Add & Norm → MLP → Add & Norm

In [ ]:
import torch.nn as nn
from usta_multi_head_attention import UstaMultiHeadAttention
from usta_layer_norm import UstaLayerNorm
from usta_mlp import UstaMLP

class UstaDecoderBlock(nn.Module):
  def __init__(self, embedding_dim, num_heads, context_length):
    super().__init__()

    self.self_attention = UstaMultiHeadAttention(embedding_dim, embedding_dim, context_length, num_heads, dropout_rate=0.1)
    self.norm1 = UstaLayerNorm(embedding_dim)
    self.mlp = UstaMLP(embedding_dim, 4 * embedding_dim)  # 4 kat genişleme
    self.norm2 = UstaLayerNorm(embedding_dim)

  def forward(self, x):
    x = self.norm1(x + self.self_attention(x))  # Add & Norm 1
    x = self.norm2(x + self.mlp(x))             # Add & Norm 2
    return x

example_tensor = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]], dtype=torch.float32)
decoder_block = UstaDecoderBlock(embedding_dim=4, num_heads=2, context_length=3)
decoder_block(example_tensor)

### Cümle üzerinde: embedding → decoder bloğu

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
sentence_meanings.shape

In [ ]:
torch.manual_seed(1)
block = UstaDecoderBlock(embedding_dim=4, num_heads=2, context_length=32)
block(sentence_meanings)

## İsteğe bağlı: Gerçek bir modelin mimarisi (Qwen3 0.6B)
Model ~1.2 GB indirir. Katmanlardaki `q_proj`, `k_proj`, `v_proj`, `o_proj` (Wq, Wk, Wv, Wo), `mlp` ve norm katmanlarına bak.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

In [ ]:
q_model